# Pipeline Financeiro — inform_27 + Danone
Carrega dados de transporte, associa ingressos Danone, enriquece com coordenadas e quilometragem, calcula métricas de rentabilidade e exporta para Parquet (Power BI).

## 1. Configuração e ligação à base de dados

In [1]:
import platform
import sqlite3
import warnings

import pandas as pd

warnings.filterwarnings("ignore")


def get_paths() -> dict:
    """Devolve os caminhos de ficheiros consoante o sistema operativo."""
    sistema = platform.system()

    if sistema == "Windows":
        return {
            "db": r"C:\Users\LISARR\Documents\python\00.DB\2026.db",
            "parquet": r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27_2026_final.parquet",
            "caminho_excel": r"C:\Users\LISARR\Documents\python\01.Financeiro\Danone_Custos_V3.xlsx",
        }

    if sistema == "Darwin":
        icloud = "/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen"
        return {
            "db": f"{icloud}/00_DB/2026.db",
            "parquet": f"{icloud}/inform_27_2026_final.parquet",
            "caminho_excel": f"{icloud}/Danone_Custos_v3.xlsx",
        }

    return {
        "db": "2026.db",
        "parquet": "inform_27_2026_final.parquet",
        "excel": "Danone_Custos_v3.xlsx",
    }


PATHS = get_paths()

with sqlite3.connect(PATHS["db"]) as con:
    df = pd.read_sql_query("SELECT * FROM inform_27_2026", con)

print(f"Linhas carregadas: {len(df):,}")
df.head()

df_excel = pd.read_excel(PATHS["caminho_excel"])
print(f"Linhas Excel: {len(df_excel):,}")
df_excel.head()

Linhas carregadas: 797,698
Linhas Excel: 44,095


,BLIINF,CODACT,PREFPE,PFEENT,PCODCL,PNOMCL,PRUTA,PCATCL,CODTLI,CODMOP,...,Mes,Año,Ruta WH,Ruta Tte. Nueva,Unnamed: 26,Unnamed: 27,Unnamed: 28,Tarifa,Ingreso,Combustible
0,FGE50PTG,11,5021758091 413963878,20260102,350150908.0,PREÇO BAIXO - DOIS AMIGOS,1734,PRE,STD,MAS,...,1,2026,Porto Prevenda,Porto Prevenda,NaN,Porto Prevenda,5021758091 413963878,31.96,1.307803,1.402227
1,FGE50PTG,11,5021765321 413964675,20260102,350151650.0,PREÇO BAIXO SUP. AV. FER. AROS,1784,PRE,STD,MAS,...,1,2026,Porto Prevenda,Porto Prevenda,NaN,Porto Prevenda,5021765321 413964675,31.96,1.186355,1.272010
2,FGE50PTG,11,5021758092 413969049,20260102,350392489.0,"SANDRA TROVISCO, UNIPESSOAL, L",1734,PRE,STD,MAS,...,1,2026,Porto Prevenda,Porto Prevenda,NaN,Porto Prevenda,5021758092 413969049,31.96,0.714626,0.766222
3,FGE50PTG,11,5021742353 413966257,20260102,350194778.0,PREÇO BAIXO - ANGEIRAS,1734,PRE,STD,MAS,...,1,2026,Porto Prevenda,Porto Prevenda,NaN,Porto Prevenda,5021742353 413966257,31.96,0.884653,0.948525
4,FGE50PTG,11,5021756292 413963116,20260102,350390901.0,FROIZ - BRAGA II- RETAIL CENTE,1734,PRE,STD,MAS,...,1,2026,Porto Prevenda,Porto Prevenda,NaN,Porto Prevenda,5021756292 413963116,31.96,3.466382,3.716654


In [2]:
# ==========================
# 1. Converter para numérico
# ==========================
df["CODACT"] = pd.to_numeric(df["CODACT"], errors="coerce")
df["INGRESODT"] = pd.to_numeric(df["INGRESODT"], errors="coerce")
df["COSTEDT"] = pd.to_numeric(df["COSTEDT"], errors="coerce")
df["PALETS"] = pd.to_numeric(df["PALETS"], errors="coerce")

# ==========================
# 2. Validar tipos
# ==========================
df[["CODACT", "INGRESODT", "COSTEDT", "PALETS"]].dtypes

CODACT       float64
INGRESODT    float64
COSTEDT      float64
PALETS       float64
dtype: object

In [3]:
# ==========================
# 1. Total antes
# ==========================
total_antes = pd.to_numeric(df["INGRESODT"], errors="coerce").fillna(0).sum()

# ==========================
# 2. Zerar CODACT 11
# ==========================
df.loc[
    pd.to_numeric(df["CODACT"], errors="coerce").eq(11),
    "INGRESODT"
] = 0

# ==========================
# 3. Total depois
# ==========================
total_depois = pd.to_numeric(df["INGRESODT"], errors="coerce").fillna(0).sum()
diferenca = total_antes - total_depois

resultado = pd.DataFrame({
    "Métrica": [
        "INGRESODT antes",
        "INGRESODT depois",
        "INGRESODT removido"
    ],
    "Valor": [
        total_antes,
        total_depois,
        diferenca
    ]
})

resultado

,Métrica,Valor
0,INGRESODT antes,30086820.68
1,INGRESODT depois,29478890.63
2,INGRESODT removido,607930.05


## 2. Dados base — `inform_27_2026`

In [4]:
# Conversão de tipos numéricos
colunas_numericas = [
    "INGRESODT", "COSTEDT", "RENTADT", "PALETSDT",
    "PESO_BRUTO", "PALETS", "KM", "KMREALES",
]
for coluna in colunas_numericas:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

df["CODEUT"] = df["CODEUT"].astype(str)

# Remover espaços em branco de todas as colunas de texto
df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

# Filtrar GESTION == "LIS", ou GESTION nulo com PROPIETARIO em CEP/PTG
df = df[
    (df["GESTION"] == "LIS")
    | (df["GESTION"].isna() & df["PROPIETARIO"].isin(["CEP", "PTG"]))
]

# Datas (após o trim, sem componente de hora)
df["FCARGA"] = pd.to_datetime(df["FCARGA"], format="%Y%m%d", errors="coerce").dt.date
df["FENTREGA"] = pd.to_datetime(df["FENTREGA"], format="%Y%m%d", errors="coerce").dt.date

df.head()

,id,PROPIETARIO,TRAYECTO,TRANSPORTISTA,TRACTORA,REMOLQUE,INGRESODT,COSTEDT,RENTADT,PALETSDT,...,LOCDES,LUGARDESCARGA,TEMP_MERC_PED,TIPOPALETA,CAMION_TIPO,CAMION_CAPACIDAD,TIPO_COMBUSTIBLE,KMREALES,ALBARAN,ficheiro_origem
93,94,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,158.09,-158.09,11.00,...,300390,NO USAR TRANP.FERNANDO SIMÕES MONTEIRO,RFG,EUR,Trailer 33 plts,33,DIESEL,145.559,NO,SAL_DAT027 (1).xls
94,95,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,5.46,-5.46,0.38,...,319558,3. MALAQUIAS - CASH & CARRY O. AZ,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,230.939,NO,SAL_DAT027 (1).xls
95,96,PTG,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,13.3,6.75,6.55,0.47,...,298515,DL Transportes Fernando Simões Monteiro Unip. Lda,TAM,EUR,Trailer 33 plts,33,DIESEL,151.803,NO,SAL_DAT027 (1).xls
96,97,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,15.95,-15.95,1.11,...,319783,"3. MARABUTO-PRODUT.ALIMENTARES,SA",RFGRFG,EUR,Trailer 33 plts,33,DIESEL,204.542,NO,SAL_DAT027 (1).xls
97,98,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,7.33,-7.33,0.51,...,319201,3. COOPERATIVA AGRICOLA DA TOCHA,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,171.983,NO,SAL_DAT027 (1).xls


## 3. Pipeline Danone — cálculo do ingresso por entrega

In [5]:
# ==========================
# 1. Ler KILOSPO
# ==========================
caminho_excel = PATHS["caminho_excel"]

raw_kilospo = pd.read_excel(caminho_excel, sheet_name="KILOSPO", header=None)
df_danone = raw_kilospo.iloc[1:].copy()
df_danone.columns = raw_kilospo.iloc[0]

# ==========================
# 2. Normalizar campos
# ==========================
df_danone["PREFPE"] = df_danone["PREFPE"].astype("string").str.strip()

if pd.api.types.is_datetime64_any_dtype(df_danone["PFEENT"]):
    df_danone["PFEENT"] = df_danone["PFEENT"].dt.strftime("%Y%m%d")
else:
    df_danone["PFEENT"] = df_danone["PFEENT"].astype("string").str.strip()

df_danone["PCODCL"] = df_danone["PCODCL"].astype("string").str.strip()
df_danone["tipo_local"] = df_danone["Ruta WH"].astype("string").str.strip()

df_danone["Ruta Tte. Nueva"] = (
    df_danone["Ruta Tte. Nueva"]
    .astype("string")
    .str.strip()
)

df_danone["Kg Neto Entregado"] = pd.to_numeric(
    df_danone["Kg Neto Entregado"],
    errors="coerce"
)

df_danone["ingresso_danone"] = pd.to_numeric(
    df_danone["Combustible"],
    errors="coerce"
)

# ==========================
# 3. Validar
# ==========================
print(f"Linhas carregadas da KILOSPO: {len(df_danone):,}")

df_danone[
    [
        "PREFPE",
        "PFEENT",
        "PCODCL",
        "PNOMCL",
        "tipo_local",
        "Ruta Tte. Nueva",
        "Kg Neto Entregado",
        "ingresso_danone",
    ]
].head()

Linhas carregadas da KILOSPO: 44,095


,PREFPE,PFEENT,PCODCL,PNOMCL,tipo_local,Ruta Tte. Nueva,Kg Neto Entregado,ingresso_danone
1,5021758091 413963878,20260102,350150908,PREÇO BAIXO - DOIS AMIGOS,Porto Prevenda,Porto Prevenda,40.92,1.402227
2,5021765321 413964675,20260102,350151650,PREÇO BAIXO SUP. AV. FER. AROS,Porto Prevenda,Porto Prevenda,37.12,1.272010
3,5021758092 413969049,20260102,350392489,"SANDRA TROVISCO, UNIPESSOAL, L",Porto Prevenda,Porto Prevenda,22.36,0.766222
4,5021742353 413966257,20260102,350194778,PREÇO BAIXO - ANGEIRAS,Porto Prevenda,Porto Prevenda,27.68,0.948525
5,5021756292 413963116,20260102,350390901,FROIZ - BRAGA II- RETAIL CENTE,Porto Prevenda,Porto Prevenda,108.46,3.716654


In [6]:
# ==========================
# 1. Ler tarifas da BD
# ==========================
with sqlite3.connect(PATHS["db"]) as con:
    df_tarifas = pd.read_sql_query(
        """
        SELECT
            tipo_local,
            tarifa_base
        FROM Danone_Tarifas_2026
        WHERE data_fim IS NULL
        """,
        con,
    )

# ==========================
# 2. Normalizar campos
# ==========================
df_tarifas["tipo_local"] = df_tarifas["tipo_local"].astype("string").str.strip()
df_tarifas["tarifa_base"] = pd.to_numeric(df_tarifas["tarifa_base"], errors="coerce")

df_danone["tipo_local"] = (
    df_danone["Ruta Tte. Nueva"]
    .astype("string")
    .str.strip()
)

# ==========================
# 3. Aplicar tarifa
# ==========================
df_danone = df_danone.drop(columns=["tarifa_base"], errors="ignore")

df_danone = df_danone.merge(
    df_tarifas,
    on="tipo_local",
    how="left",
    validate="many_to_one",
)

# ==========================
# 4. Validar
# ==========================
print(f"Linhas: {len(df_danone):,}")
print(f"Com tarifa: {df_danone['tarifa_base'].notna().sum():,}")
print(f"Sem tarifa: {df_danone['tarifa_base'].isna().sum():,}")

df_danone[
    ["PREFPE", "Ruta Tte. Nueva", "tipo_local", "tarifa_base"]
].head(20)

Linhas: 44,095
Com tarifa: 30,793
Sem tarifa: 13,302


,PREFPE,Ruta Tte. Nueva,tipo_local,tarifa_base
0,5021758091 413963878,Porto Prevenda,Porto Prevenda,31.96000
1,5021765321 413964675,Porto Prevenda,Porto Prevenda,31.96000
2,5021758092 413969049,Porto Prevenda,Porto Prevenda,31.96000
3,5021742353 413966257,Porto Prevenda,Porto Prevenda,31.96000
4,5021756292 413963116,Porto Prevenda,Porto Prevenda,31.96000
5,5021762387 413967065,Porto Prevenda,Porto Prevenda,31.96000
6,5021726862 413962594,Porto Prevenda,Porto Prevenda,31.96000
7,5021766864 413973924,Porto Prevenda,Porto Prevenda,31.96000
8,5021767276 413970112,Porto Prevenda,Porto Prevenda,31.96000
9,5021741336 413972570,Porto Prevenda,Porto Prevenda,31.96000


In [7]:
# ==========================
# 1. Calcular ingresso Danone
# ==========================
df_danone["Kg Neto Entregado"] = pd.to_numeric(
    df_danone["Kg Neto Entregado"],
    errors="coerce"
)

df_danone["ingresso_danone"] = (
    df_danone["Kg Neto Entregado"] / 1000
) * df_danone["tarifa_base"]

# ==========================
# 2. Validar cálculo
# ==========================
df_danone[
    [
        "PREFPE",
        "tipo_local",
        "Kg Neto Entregado",
        "tarifa_base",
        "ingresso_danone",
    ]
].head(20)

,PREFPE,tipo_local,Kg Neto Entregado,tarifa_base,ingresso_danone
0,5021758091 413963878,Porto Prevenda,40.920,31.96000,1.307803
1,5021765321 413964675,Porto Prevenda,37.120,31.96000,1.186355
2,5021758092 413969049,Porto Prevenda,22.360,31.96000,0.714626
3,5021742353 413966257,Porto Prevenda,27.680,31.96000,0.884653
4,5021756292 413963116,Porto Prevenda,108.460,31.96000,3.466382
5,5021762387 413967065,Porto Prevenda,113.750,31.96000,3.635450
6,5021726862 413962594,Porto Prevenda,29.000,31.96000,0.926840
7,5021766864 413973924,Porto Prevenda,143.040,31.96000,4.571558
8,5021767276 413970112,Porto Prevenda,32.660,31.96000,1.043814
9,5021741336 413972570,Porto Prevenda,31.900,31.96000,1.019524


In [8]:
sem_classificacao = df_danone[df_danone["tipo_local"].isna()]

lista_referencias_sem_classificacao = (
    sem_classificacao[["PCODCL", "PNOMCL", "PREFPE", "PFEENT"]]
    .sort_values(["PCODCL", "PFEENT"])
    .reset_index(drop=True)
)

print(f"Linhas sem tipo_local: {len(lista_referencias_sem_classificacao):,}")
lista_referencias_sem_classificacao

Linhas sem tipo_local: 258


,PCODCL,PNOMCL,PREFPE,PFEENT
0,350244321,"JNR - DISTRIBUICAO ALIMENTAR,",5021958305 414184555,20260209
1,350369679,PD MATOSINHOS SUL,5022148143 414337557,20260303
2,350394558,CNT PORTIMÃO 2,5022122820 414349606,20260303
3,350407940,ITM TORRES NOVAS,5021952613 414167617,20260202
4,350407940,ITM TORRES NOVAS,5022559594 414769203,20260420
...,...,...,...,...
253,350494624,Rilhadas Turismo,5023553123 415669317,20260813
254,<NA>,TESTE,teste recheio camarate,20260721
255,<NA>,TESTE,Teste,20260828
256,<NA>,TESTE ETI PD,TESTE,20260828


In [9]:
# ==========================
# 3. Ingresso por mês
# ==========================
df_danone["PFEENT"] = pd.to_datetime(
    df_danone["PFEENT"],
    errors="coerce"
)

df_danone["mes"] = df_danone["PFEENT"].dt.month

soma_por_mes = (
    df_danone
    .groupby("mes", as_index=False)
    .agg(
        linhas=("ingresso_danone", "size"),
        ingresso_total=("ingresso_danone", "sum")
    )
)

soma_por_mes

,mes,linhas,ingresso_total
0,1,5522,120737.811470
1,2,5387,110624.537493
2,3,5730,130474.644257
3,4,5309,132493.699391
4,5,5293,129458.216679
5,6,5405,142437.495322
6,7,6072,165604.056730
7,8,5159,145339.525013
8,9,218,3867.446232


In [10]:
import calendar

mes8 = df_danone[df_danone["mes"] == 8]

dias_com_dados = mes8["PFEENT"].dt.day.nunique()
dias_total_mes = calendar.monthrange(2026, 8)[1]
total_mes8 = mes8["ingresso_danone"].sum(min_count=1)
media_diaria = total_mes8 / dias_com_dados
projecao_mes8 = media_diaria * dias_total_mes

print(f"Dias com dados em agosto: {dias_com_dados}")
print(f"Dias totais no mês: {dias_total_mes}")
print(f"Total registado até agora: {total_mes8:,.0f}")
print(f"Média diária: {media_diaria:,.0f}")
print(f"Projeção para o mês completo: {projecao_mes8:,.0f}")

Dias com dados em agosto: 26
Dias totais no mês: 31
Total registado até agora: 145,340
Média diária: 5,590
Projeção para o mês completo: 173,289


## 4. Associar o ingresso Danone ao dataframe principal

In [11]:
# Somar ingresso Danone por entrega (PREFPE + PFEENT)
df_ingresso_danone = df_danone[["PREFPE", "PFEENT", "ingresso_danone"]].copy()

# PREFPE fica inteiro (não se divide por espaços) — REFERENCIA no df guarda
# os dois códigos colados no mesmo formato ("5021758091     413963878"),
# por isso a chave de correspondência tem de ficar igual dos dois lados.
df_ingresso_danone["PREFPE"] = df_ingresso_danone["PREFPE"].astype("string").str.strip()

# PFEENT pode já vir como data (Timestamp) ou como texto "AAAAMMDD", consoante a fonte.
if pd.api.types.is_datetime64_any_dtype(df_ingresso_danone["PFEENT"]):
    df_ingresso_danone["PFEENT"] = pd.to_datetime(df_ingresso_danone["PFEENT"])
else:
    df_ingresso_danone["PFEENT"] = pd.to_datetime(
        df_ingresso_danone["PFEENT"].astype("string").str.strip(), format="%Y%m%d", errors="coerce"
    )

df_ingresso_danone = (
    df_ingresso_danone
    .groupby(["PREFPE", "PFEENT"], dropna=False)
    .agg(ingresso_danone_total=("ingresso_danone", lambda x: x.sum(min_count=1)))
    .reset_index()
)

# Preparar chaves no df principal
df["REFERENCIA"] = df["REFERENCIA"].astype("string").str.strip()
df["FENTREGA"] = pd.to_datetime(df["FENTREGA"], errors="coerce")
df["_ordem_original"] = range(len(df))

df = df.merge(
    df_ingresso_danone,
    left_on=["REFERENCIA", "FENTREGA"],
    right_on=["PREFPE", "PFEENT"],
    how="left",
    validate="many_to_one",
).sort_values("_ordem_original").reset_index(drop=True)

# Uma entrega pode ter várias linhas em df (mesma REFERENCIA + FENTREGA);
# o ingresso só é atribuído à última linha, para não o contar em duplicado.
ultima_linha = ~df.duplicated(subset=["REFERENCIA", "FENTREGA"], keep="last")

df["ingresso_danone"] = pd.Series(pd.NA, index=df.index, dtype="Float64")
df.loc[ultima_linha, "ingresso_danone"] = df.loc[ultima_linha, "ingresso_danone_total"]

df = df.drop(columns=["PREFPE", "PFEENT", "ingresso_danone_total", "_ordem_original"])

# Ingresso total = ingresso do transporte (INGRESODT) + ingresso Danone
df["total_ingresso"] = df["INGRESODT"].fillna(0) + df["ingresso_danone"].fillna(0)

diferenca = df_danone["ingresso_danone"].sum(min_count=1) - df["ingresso_danone"].sum(min_count=1)
print(f"Diferença entre ingresso Danone calculado e associado: {diferenca:,.2f}")

Diferença entre ingresso Danone calculado e associado: 8,757.44


In [12]:
df["mes"] = df["FENTREGA"].dt.month

soma_por_mes_df = (
    df
    .groupby("mes")
    .agg(ingresso_total=("ingresso_danone", lambda x: x.sum(min_count=1)))
    .reset_index()
)
soma_por_mes_df["ingresso_total"] = soma_por_mes_df["ingresso_total"].round(0).map("{:,.0f}".format)
soma_por_mes_df

,mes,ingresso_total
0,1,"119,984"
1,2,"110,077"
2,3,"129,981"
3,4,"131,881"
4,5,"128,984"
5,6,"139,378"
6,7,"163,673"
7,8,"144,484"
8,9,"3,839"


In [26]:
resumo_por_atividade_mes = (
    df[df["CODACT"].isin(["11", "13", "311"])]
    .groupby(["CODACT", "mes"])
    .agg(
        soma_ingresodt=("INGRESODT", "sum"),
        soma_ingresso_danone=("ingresso_danone", lambda x: x.sum(min_count=1)),
    )
    .reset_index()
)

resumo_por_atividade_mes["soma_ingresodt"] = resumo_por_atividade_mes["soma_ingresodt"].round(0).map("{:,.0f}".format)
resumo_por_atividade_mes["soma_ingresso_danone"] = resumo_por_atividade_mes["soma_ingresso_danone"].round(0).map("{:,.0f}".format)

resumo_por_atividade_mes

,CODACT,mes,soma_ingresodt,soma_ingresso_danone


## 5. Features derivadas

In [14]:
# Data, semana e dia da semana
df["data"] = pd.to_datetime(df["FCARGA"])
df["week_number"] = df["data"].dt.isocalendar().week
df["week_day"] = df["data"].dt.day_name()
df["mes"] = df["data"].dt.month
df["mes_nome"] = df["data"].dt.month_name()

df[["FCARGA", "week_number", "week_day", "mes_nome", "total_ingresso"]].head()

,FCARGA,week_number,week_day,mes_nome,total_ingresso
0,2026-02-01,5,Sunday,February,0.0
1,2026-02-01,5,Sunday,February,31.332933
2,2026-02-01,5,Sunday,February,13.3
3,2026-02-01,5,Sunday,February,105.402351
4,2026-02-01,5,Sunday,February,42.456427


In [15]:
# Normalização da capacidade do camião (agrupar capacidades equivalentes)
mapeamento_capacidade = {
    4: 6, 5: 6, 6: 6,
    8: 12, 12: 12,
    14: 20, 15: 20, 16: 20, 18: 20, 20: 20,
    22: 24, 24: 24,
    33: 33, 66: 66,
}

df["CAMION_CAPACIDAD_NUM"] = pd.to_numeric(df["CAMION_CAPACIDAD"], errors="coerce")
df["capacidade_norm"] = df["CAMION_CAPACIDAD_NUM"].map(mapeamento_capacidade)

# Capacidade em falta (valor 0): preencher com a capacidade mais comum da mesma rota
linhas_sem_capacidade = df["CAMION_CAPACIDAD_NUM"] == 0
if linhas_sem_capacidade.any():
    capacidade_por_rota = (
        df.loc[df["CAMION_CAPACIDAD_NUM"] > 0]
        .groupby("CODEUT")["capacidade_norm"]
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.max())
    )
    for rota in df.loc[linhas_sem_capacidade, "CODEUT"].unique():
        if rota in capacidade_por_rota.index:
            df.loc[(df["CODEUT"] == rota) & linhas_sem_capacidade, "capacidade_norm"] = capacidade_por_rota[rota]

df["dados_validos"] = df["capacidade_norm"].notna()
print(f"Linhas com capacidade válida: {df['dados_validos'].sum():,} / {len(df):,}")

Linhas com capacidade válida: 120,095 / 126,837


## 6. Coordenadas geográficas (origem e destino)

In [16]:
with sqlite3.connect(PATHS["db"]) as con:
    coords = pd.read_sql_query(
        "SELECT cp AS CP, point_x AS POINT_X, point_y AS POINT_Y FROM coordenadas",
        con,
    )

coords["CP"] = coords["CP"].astype("string").str.strip()
coords["POINT_X"] = pd.to_numeric(coords["POINT_X"], errors="coerce")
coords["POINT_Y"] = pd.to_numeric(coords["POINT_Y"], errors="coerce")
coords["CP_parte"] = coords["CP"].str.split("-").str[0]

# Centroide por prefixo de código postal — usado como fallback quando o CP completo não tem match
centroid = (
    coords.groupby("CP_parte")[["POINT_X", "POINT_Y"]]
    .mean()
    .reset_index()
    .rename(columns={"POINT_X": "longitude_centroid", "POINT_Y": "latitude_centroid"})
)

coords_origem = coords[["CP", "POINT_X", "POINT_Y"]].rename(
    columns={"CP": "CPOSTAL", "POINT_X": "longitude_origem", "POINT_Y": "latitude_origem"}
)
coords_destino = coords[["CP", "POINT_X", "POINT_Y"]].rename(
    columns={"CP": "CPOSTAD", "POINT_X": "longitude_destino", "POINT_Y": "latitude_destino"}
)

df["CPOSTAL_parte"] = df["CPOSTAL"].str.split("-").str[0]
df["CPOSTAD_parte"] = df["CPOSTAD"].str.split("-").str[0]

df = df.merge(coords_origem, on="CPOSTAL", how="left")
df = df.merge(coords_destino, on="CPOSTAD", how="left")

# Preencher falhas de match com o centroide do prefixo do código postal
centroid_origem = centroid.rename(columns={
    "CP_parte": "CPOSTAL_parte",
    "longitude_centroid": "longitude_origem_c",
    "latitude_centroid": "latitude_origem_c",
})
df = df.merge(centroid_origem, on="CPOSTAL_parte", how="left")
df["longitude_origem"] = df["longitude_origem"].fillna(df["longitude_origem_c"])
df["latitude_origem"] = df["latitude_origem"].fillna(df["latitude_origem_c"])

centroid_destino = centroid.rename(columns={
    "CP_parte": "CPOSTAD_parte",
    "longitude_centroid": "longitude_destino_c",
    "latitude_centroid": "latitude_destino_c",
})
df = df.merge(centroid_destino, on="CPOSTAD_parte", how="left")
df["longitude_destino"] = df["longitude_destino"].fillna(df["longitude_destino_c"])
df["latitude_destino"] = df["latitude_destino"].fillna(df["latitude_destino_c"])

df = df.drop(columns=[
    "CPOSTAL_parte", "CPOSTAD_parte",
    "longitude_origem_c", "latitude_origem_c",
    "longitude_destino_c", "latitude_destino_c",
])

print(f"Matches origem: {df['longitude_origem'].notna().sum():,}")
print(f"Matches destino: {df['longitude_destino'].notna().sum():,}")


Matches origem: 0
Matches destino: 0


## 7. Métricas de rentabilidade

In [17]:
df["custo_por_palete"] = df["COSTEDT"] / df["PALETS"].replace(0, 1)
df["ingresso_por_palete"] = df["total_ingresso"] / df["PALETS"].replace(0, 1)

# Taxa de ocupação por rota (CODEUT): total de paletes da rota vs capacidade do veículo
rota_totais = (
    df.groupby("CODEUT")
    .agg(total_palets=("PALETS", "sum"), capacidade_rota=("capacidade_norm", "first"))
    .reset_index()
)
rota_totais["taxa_rota"] = rota_totais["total_palets"] / rota_totais["capacidade_rota"].replace(0, 1) * 100

df = df.merge(rota_totais[["CODEUT", "taxa_rota", "total_palets"]], on="CODEUT", how="left")
df["taxa_ocupacao"] = df["PALETS"] / df["total_palets"].replace(0, 1) * df["taxa_rota"]
df = df.drop(columns=["taxa_rota", "total_palets"])

df["margem"] = df["total_ingresso"] - df["COSTEDT"]
df["margem_por_palete"] = df["ingresso_por_palete"] - df["custo_por_palete"]

df.head()

,id,PROPIETARIO,TRAYECTO,TRANSPORTISTA,TRACTORA,REMOLQUE,INGRESODT,COSTEDT,RENTADT,PALETSDT,...,dados_validos,longitude_origem,latitude_origem,longitude_destino,latitude_destino,custo_por_palete,ingresso_por_palete,taxa_ocupacao,margem,margem_por_palete
0,94,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,158.09,-158.09,11.00,...,True,NaN,NaN,NaN,NaN,14.371818,0.0,33.333333,-158.09,-14.371818
1,95,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,5.46,-5.46,0.38,...,True,NaN,NaN,NaN,NaN,5.460000,31.332933,3.030303,25.872933,25.872933
2,96,PTG,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,13.3,6.75,6.55,0.47,...,True,NaN,NaN,NaN,NaN,6.750000,13.3,3.030303,6.55,6.55
3,97,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,15.95,-15.95,1.11,...,True,NaN,NaN,NaN,NaN,7.975000,52.701176,6.060606,89.452351,44.726176
4,98,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,7.33,-7.33,0.51,...,True,NaN,NaN,NaN,NaN,7.330000,42.456427,3.030303,35.126427,35.126427


## 8. Selecionar colunas finais e exportar

In [18]:
colunas_finais = [
    "REFERENCIA", "FENTREGA", "CODEUT", "capacidade_norm", "PALETS", "INGRESODT", "ingresso_danone","total_ingresso", "COSTEDT",
    "PROV_ORIGEN", "LOCORIGEN", "PROV_DESTINO", "LOCDESTINO", "CPOSTAL", "CPOSTAD",
    "longitude_origem", "latitude_origem", "longitude_destino", "latitude_destino",
    "TRANSPORTISTA", "week_day", "week_number", "mes", "mes_nome", "data",
    "dados_validos", "LOCCAR", "LUGARCARGA", "LOCDES", "LUGARDESCARGA", "TRACTORA",
    "PESO_BRUTO", "CODEDT", "CODACT", "TIPOCLIENTE", "TIPOFLUJO",
    "PROV_ENTREGAR", "PAISENTREGAR", "ACTIVIDAD",
    "custo_por_palete", "ingresso_por_palete", "taxa_ocupacao", "margem", "margem_por_palete",
]

df_sel = df[colunas_finais].copy()
df_sel.head()

,REFERENCIA,FENTREGA,CODEUT,capacidade_norm,PALETS,INGRESODT,ingresso_danone,total_ingresso,COSTEDT,PROV_ORIGEN,...,TIPOCLIENTE,TIPOFLUJO,PROV_ENTREGAR,PAISENTREGAR,ACTIVIDAD,custo_por_palete,ingresso_por_palete,taxa_ocupacao,margem,margem_por_palete
0,COIMBRA 20/29+70/79-02.02,2026-02-02,3365589,33.0,11.0,0.0,<NA>,0.0,158.09,Lisboa,...,NaN,Directo,ANADIA,PORTUGAL,DANONE PORTUGAL CAPILAR,14.371818,0.0,33.333333,-158.09,-14.371818
1,5021950798 414190635,2026-02-02,3365589,33.0,1.0,0.0,31.332933,31.332933,5.46,Lisboa,...,TLD PORTUGAL,Directo,Aveiro,PORTUGAL,DANONE PORTUGAL,5.460000,31.332933,3.030303,25.872933,25.872933
2,2021792046,2026-02-02,3365589,33.0,1.0,13.3,<NA>,13.3,6.75,Lisboa,...,NaN,Directo,Coimbra,PORTUGAL,SUMOLCOMPAL MARKETING,6.750000,13.3,3.030303,6.55,6.55
3,5021982094 414203894,2026-02-02,3365589,33.0,2.0,0.0,105.402351,105.402351,15.95,Lisboa,...,TLD PORTUGAL,Directo,Aveiro,PORTUGAL,DANONE PORTUGAL,7.975000,52.701176,6.060606,89.452351,44.726176
4,5021963392 414182550,2026-02-02,3365589,33.0,1.0,0.0,42.456427,42.456427,7.33,Lisboa,...,TLD PORTUGAL,Directo,Coimbra,PORTUGAL,DANONE PORTUGAL,7.330000,42.456427,3.030303,35.126427,35.126427


In [19]:
df_sel.to_parquet(PATHS["parquet"], index=False)

print(f"Ficheiro exportado: {PATHS['parquet']}")
print(f"Linhas: {len(df_sel):,}")
print(f"Colunas: {len(df_sel.columns)}")
print(f"Tamanho: {df_sel.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Ficheiro exportado: C:\Users\LISARR\Documents\python\01.Financeiro\inform_27_2026_final.parquet
Linhas: 126,837
Colunas: 44
Tamanho: 80.78 MB


# Analises


In [20]:
# Marcar entregas com ingresso_danone atribuído (fazem match ao Excel Danone)
df["tem_danone"] = df.groupby(["REFERENCIA", "FENTREGA"])["ingresso_danone"].transform(lambda x: x.notna().any())

resumo_danone_mes = (
    df[df["CODACT"].isin(["11"]) & df["tem_danone"]]
    .groupby("mes")
    .agg(
        ingresso=("ingresso_danone", "sum"),
        custo=("COSTEDT", "sum"),
        paletes=("PALETS", "sum"),
        peso=("PESO_BRUTO", "sum"),
    )
    .reset_index()
)
resumo_danone_mes["margem"] = resumo_danone_mes["ingresso"] - resumo_danone_mes["custo"]

for coluna in ["ingresso", "custo", "margem", "paletes", "peso"]:
    resumo_danone_mes[coluna] = resumo_danone_mes[coluna].round(0).map("{:,.0f}".format)

resumo_danone_mes

,mes,ingresso,custo,paletes,peso,margem


In [21]:
resumo_311_mes = (
    df[df["CODACT"].isin(["311"])]
    .groupby("mes")
    .agg(
        ingresso=("INGRESODT", "sum"),
        custo=("COSTEDT", "sum"),
        paletes=("PALETS", "sum"),
        peso=("PESO_BRUTO", "sum"),
    )
    .reset_index()
)
resumo_311_mes["margem"] = resumo_311_mes["ingresso"] - resumo_311_mes["custo"]

for coluna in ["ingresso", "custo", "margem", "paletes", "peso"]:
    resumo_311_mes[coluna] = resumo_311_mes[coluna].round(0).map("{:,.0f}".format)

resumo_311_mes

,mes,ingresso,custo,paletes,peso,margem


In [22]:
resumo_311_mes = (
    df[df["CODACT"].isin(["074"])]
    .groupby("mes")
    .agg(
        ingresso=("INGRESODT", "sum"),
        custo=("COSTEDT", "sum"),
        paletes=("PALETS", "sum"),
        peso=("PESO_BRUTO", "sum"),
    )
    .reset_index()
)
resumo_311_mes["margem"] = resumo_311_mes["ingresso"] - resumo_311_mes["custo"]

for coluna in ["ingresso", "custo", "margem", "paletes", "peso"]:
    resumo_311_mes[coluna] = resumo_311_mes[coluna].round(0).map("{:,.0f}".format)

resumo_311_mes

,mes,ingresso,custo,paletes,peso,margem


In [23]:
resumo_outros_mes = (
    df[~df["CODACT"].isin(["011", "311", "013", "074"])]
    .groupby("mes")
    .agg(
        ingresso=("INGRESODT", "sum"),
        custo=("COSTEDT", "sum"),
        paletes=("PALETS", "sum"),
        peso=("PESO_BRUTO", "sum"),
    )
    .reset_index()
)
resumo_outros_mes["margem"] = resumo_outros_mes["ingresso"] - resumo_outros_mes["custo"]

for coluna in ["ingresso", "custo", "margem", "paletes", "peso"]:
    resumo_outros_mes[coluna] = resumo_outros_mes[coluna].round(0).map("{:,.0f}".format)

resumo_outros_mes

,mes,ingresso,custo,paletes,peso,margem
0,1,"310,438","464,003","49,345","10,579,447","-153,565"
1,2,"280,717","422,153","42,086","9,132,658","-141,436"
2,3,"337,785","525,819","52,061","13,352,482","-188,034"
3,4,"344,905","535,635","51,922","11,796,200","-190,729"
4,5,"340,001","532,012","50,522","11,871,848","-192,011"
5,6,"335,489","549,940","49,415","12,446,435","-214,451"
6,7,"407,362","601,524","52,460","13,619,063","-194,162"
7,8,"360,878","553,101","52,085","12,789,029","-192,222"
8,9,"24,931","33,762","3,940","751,545","-8,831"
9,12,"22,722","32,011","2,354","581,316","-9,289"


In [24]:
resumo_total_sem_013 = (
    df[~df["CODACT"].isin(["013"])]
    .groupby("mes")
    .agg(
        ingresso=("total_ingresso", "sum"),
        custo=("COSTEDT", "sum"),
        paletes=("PALETS", "sum"),
        peso=("PESO_BRUTO", "sum"),
    )
    .reset_index()
)
resumo_total_sem_013["margem"] = resumo_total_sem_013["ingresso"] - resumo_total_sem_013["custo"]

for coluna in ["ingresso", "custo", "margem", "paletes", "peso"]:
    resumo_total_sem_013[coluna] = resumo_total_sem_013[coluna].round(0).map("{:,.0f}".format)

resumo_total_sem_013

,mes,ingresso,custo,paletes,peso,margem
0,1,"429,817","464,003","49,345","10,579,447","-34,186"
1,2,"390,086","422,153","42,086","9,132,658","-32,067"
2,3,"472,129","525,819","52,061","13,352,482","-53,690"
3,4,"478,429","535,635","51,922","11,796,200","-57,205"
4,5,"466,178","532,012","50,522","11,871,848","-65,834"
5,6,"479,427","549,940","49,415","12,446,435","-70,512"
6,7,"565,705","601,524","52,460","13,619,063","-35,819"
7,8,"501,742","553,101","52,085","12,789,029","-51,359"
8,9,"25,385","33,762","3,940","751,545","-8,377"
9,12,"28,609","32,011","2,354","581,316","-3,402"


In [25]:
# DF com as linhas "buraco": CODACT=011 sem match ao Excel Danone (custo sem contrapartida de receita)
df["tem_danone"] = df.groupby(["REFERENCIA", "FENTREGA"])["ingresso_danone"].transform(lambda x: x.notna().any())

df_buraco_011 = df[(df["CODACT"] == "011") & (~df["tem_danone"])].copy()

df_buraco_011 = df_buraco_011[[
    "REFERENCIA", "FENTREGA", "CODEUT", "TRANSPORTISTA",
    "LOCORIGEN", "LOCDESTINO", "PALETS", "INGRESODT", "COSTEDT", "mes"
]]

df_buraco_011.head()

,REFERENCIA,FENTREGA,CODEUT,TRANSPORTISTA,LOCORIGEN,LOCDESTINO,PALETS,INGRESODT,COSTEDT,mes
